# RAG : Usando FAISS como base de conocimiento

In [1]:
import os, getpass

Faiss es una biblioteca para la búsqueda eficiente de similitudes y agrupación de vectores. Contiene algoritmos que buscan en conjuntos de vectores de cualquier tamaño, hasta aquellos que posiblemente no quepan en la RAM. También contiene código de soporte para evaluación y ajuste de parámetros.

Faiss está escrito en C++ con contenedores completos para Python. Algunos de los algoritmos más útiles se implementan en la GPU. Está desarrollado por Facebook AI Research .

## Generando fragmentos de un archivo PDF

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

loader = PyPDFLoader("pdf/DataAI-ExecutiveLeadership-2024.pdf") #22 páginas
data = loader.load()

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 400, 
    chunk_overlap = 20
)

docs = text_splitter.split_documents(documents = data)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
len(docs), docs

(22,
 [Document(metadata={'producer': 'GPL Ghostscript 10.00.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': "D:20240102085016Z00'00'", 'moddate': "D:20240102085016Z00'00'", 'source': 'pdf/DataAI-ExecutiveLeadership-2024.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1'}, page_content='The State of Data and AI in \nLeading Companies 2024\nWith a Foreword by Randy Bean  \nand Thomas H. Davenport\nEXECUTIVE SUMMARY OF FINDINGS\n2024 DATA AND AI LEADERSHIP  \nEXECUTIVE SURVEY\n2024 © WAVESTONE'),
  Document(metadata={'producer': 'GPL Ghostscript 10.00.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': "D:20240102085016Z00'00'", 'moddate': "D:20240102085016Z00'00'", 'source': 'pdf/DataAI-ExecutiveLeadership-2024.pdf', 'total_pages': 22, 'page': 1, 'page_label': '2'}, page_content='2\nFOREWORD\nThe past year has been an extraordinary \none in many respects, not the least of \nwhich is the amazing rise of Generative AI. \nThat overshadows any other develo

## Cargando datos a FAISS

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 607.58it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu API Key de OpenAI : ")

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

template = """Responde a la pregunta basándote solo en el siguiente contexto.
Si no puedes responder a la pregunta, responde exactamente:
"No lo sé disculpa, puedes buscar la información en internet."

<context>
{context}
</context>

Pregunta: {input}
Respuesta:
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "input"]
)

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.0
)

combine_docs_chain = create_stuff_documents_chain(llm, prompt)

qa = create_retrieval_chain(
    vectorstore.as_retriever(),
    combine_docs_chain
)


In [10]:
qa.invoke({"input": "¿de qué trata el documento?"})["answer"]

'El documento trata sobre el resumen de hallazgos de la encuesta ejecutiva de liderazgo en datos y AI de 2024 de Wavestone.'

In [12]:
qa.invoke({"input": "bríndame un resumen del documento"})["answer"]

'El documento presenta los hallazgos de una encuesta ejecutiva sobre liderazgo en datos y AI en el año 2024. Se destaca que el rol de Chief Data Officer (CDO) o Chief Data and Analytics Officer (CDAO) es considerado necesario por las empresas, aunque la rotación en estos roles ha sido alta y las tenencias cortas. Se menciona que la responsabilidad principal de datos y análisis está cada vez más en manos de los CDO/CDAO, y se discute sobre la evolución del rol, los desafíos principales para convertirse en una organización impulsada por datos, y la importancia de la inteligencia artificial generativa. También se menciona que el rol de CDO/CDAO seguirá evolucionando y enfrentando transformaciones en el futuro.'